# Release16 Holdout Slice Repair Candidate Screen

This notebook turns the current frontier blocker into a scored repair contract. It compares the existing official warm-start repair cards and decides whether any candidate is allowed to advance to reward/checkpoint screening.

Governance fields:
- Baselines: canonical `soxlbacktest_qlib_walkforward_baseline_v1`; direct comparison `release16_joint_target_holdout_slice_repair_v1`.
- Role: `diagnose`.
- Target window: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Changed layer: `model_selection`, `bc_warm_start`, `selection`.
- External reference: existing release16 action-head / aux-state-loss repair evidence and the SOXLBacktest qlib baseline.
- Closest historical success: `cap10_three_fold_bc_slice_screen_v1` proved train-slice BC learnability, but did not provide official replay promotion.
- Closest historical failure: `release16_downtrend_drawdown_objective_screen_v1` repaired drawdown but kept holdout sparse-slice failure and a `wf_2020` relative-return gap.
- Why this is not a duplicate: this notebook adds a shared holdout-repair scoring interface and ranks multiple repair cards into a single next training contract instead of running another replay.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_bc import (
    score_state_objective_holdout_repair_candidates,
    summarize_state_objective_holdout_repair_screen,
)

EXPERIMENT = "release16_holdout_slice_repair_candidate_screen_v1"
OUT_DIR = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)
CARD_ROOT = ROOT / "runtime" / "model_selection" / "card"

REQUIRED_FOLDS = ("wf_2020", "wf_2022", "wf_2023_2024")
TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0

SOURCE_CARDS = [
    "release16_joint_drawdown_aux_state_head_bc_holdout_replay_v1",
    "release16_action_head_position_loss_screen_v1",
    "release16_downtrend_drawdown_objective_screen_v1",
    "release16_joint_target_holdout_slice_repair_v1",
]


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _source_candidate(row: dict) -> str:
    return str(row.get("selected_candidate") or row.get("source_holdout_selected_candidate") or "")


def rows_from_summary(card: str, summary: dict) -> list[dict[str, object]]:
    rows = []
    selected = summary.get("selected_by_fold", {})
    if not isinstance(selected, dict):
        return rows
    for fold in REQUIRED_FOLDS:
        payload = selected.get(fold)
        if not isinstance(payload, dict):
            continue
        rows.append(
            {
                "source_label": card,
                "fold": fold,
                "candidate": _source_candidate(payload),
                "relative_return": payload.get("selected_relative_return"),
                "max_dd": payload.get("selected_max_dd"),
                "avg_pos": payload.get("selected_avg_pos"),
                "holdout_slice_gate": bool(payload.get("holdout_slice_gate", payload.get("selected_holdout_slice_gate", False))),
                "failed_slices": str(payload.get("failed_slices", payload.get("holdout_failed_slices", ""))),
                "train_slice_gate": bool(payload.get("train_slice_gate", payload.get("selected_train_slice_gate", False))),
                "drift_gate": bool(payload.get("drift_gate", payload.get("selected_drift_gate", False))),
                "candidate_gate": bool(payload.get("selected_candidate_gate", False)),
                "failure_type": str(payload.get("failure_type", "")),
            }
        )
    return rows


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
summaries = {card: read_json(CARD_ROOT / card / "summary.json") for card in SOURCE_CARDS}
raw_rows = []
for card, payload in summaries.items():
    raw_rows.extend(rows_from_summary(card, payload))
repair_rows = pd.DataFrame(raw_rows)
repair_rows.to_csv(OUT_DIR / "candidate_repair_rows.csv", index=False)

scores = score_state_objective_holdout_repair_candidates(
    repair_rows,
    required_folds=REQUIRED_FOLDS,
    target_relative_return_pp=TARGET_RELATIVE_RETURN_PP,
    target_max_dd_pct=TARGET_MAX_DD_PCT,
)
scores.to_csv(OUT_DIR / "holdout_repair_scores.csv", index=False)
summary_core = summarize_state_objective_holdout_repair_screen(scores)

scores


,source_label,fold_count,missing_folds,target_gate,holdout_slice_gate,holdout_repair_gate,target_gate_pass_count,holdout_slice_gate_pass_count,fold_repair_gate_pass_count,min_relative_return,max_max_dd,relative_gap_loss,max_dd_gap_loss,holdout_slice_loss,missing_fold_loss,repair_loss,failed_folds,failed_slices,required_repair_terms
0,release16_downtrend_drawdown_objective_screen_v1,3,,False,False,False,2,0,0,-5.602809,17.693927,35.602809,0.000000,75.0,0.0,110.602809,"wf_2020,wf_2022,wf_2023_2024","aux_state_head_release_state,failedfold_state_...","[checkpoint_objective_loss, drawdown_state_aux..."
1,release16_joint_drawdown_aux_state_head_bc_hol...,3,,False,False,False,0,0,0,-5.315300,29.516629,35.547380,104.486933,75.0,0.0,215.034313,"wf_2020,wf_2022,wf_2023_2024","all,drawdown_control_state,is_downtrend_state,...","[checkpoint_objective_loss, drawdown_state_aux..."
2,release16_action_head_position_loss_screen_v1,3,,False,False,False,1,1,0,-13.659859,28.344218,43.659859,139.250264,50.0,0.0,232.910123,"wf_2020,wf_2022,wf_2023_2024","drawdown_control_state,is_downtrend_state","[checkpoint_objective_loss, drawdown_state_aux..."
3,release16_joint_target_holdout_slice_repair_v1,3,,False,False,False,0,0,0,-264.728894,26.832912,294.728894,106.755442,75.0,0.0,476.484336,"wf_2020,wf_2022,wf_2023_2024","drawdown_control_state,is_downtrend_state,rele...","[checkpoint_objective_loss, drawdown_state_aux..."


In [3]:
selected_source = str(summary_core.get("selected_source", ""))
selected_source_rows = repair_rows[repair_rows["source_label"].eq(selected_source)].copy()
selected_by_fold = {}
for row in selected_source_rows.to_dict("records"):
    fold = str(row["fold"])
    rel = pd.to_numeric(pd.Series([row.get("relative_return")]), errors="coerce").iloc[0]
    dd = pd.to_numeric(pd.Series([row.get("max_dd")]), errors="coerce").iloc[0]
    relative_gap = max(0.0, TARGET_RELATIVE_RETURN_PP - float(rel)) if pd.notna(rel) else None
    dd_gap = max(0.0, float(dd) - TARGET_MAX_DD_PCT) if pd.notna(dd) else None
    selected_by_fold[fold] = {
        "selected_candidate": str(row.get("candidate", "")),
        "selected_relative_return": None if pd.isna(rel) else float(rel),
        "selected_max_dd": None if pd.isna(dd) else float(dd),
        "selected_avg_pos": None if pd.isna(row.get("avg_pos")) else float(row.get("avg_pos")),
        "holdout_slice_gate": bool(row.get("holdout_slice_gate", False)),
        "train_slice_gate": bool(row.get("train_slice_gate", False)),
        "drift_gate": bool(row.get("drift_gate", False)),
        "selected_candidate_gate": bool(row.get("candidate_gate", False)),
        "failed_slices": str(row.get("failed_slices", "")),
        "failure_type": str(row.get("failure_type", "")),
        "relative_gap_to_target_pp": relative_gap,
        "max_dd_gap_to_target_pp": dd_gap,
    }

selected_score = scores.iloc[0].to_dict() if not scores.empty else {}
created_at = datetime.now(timezone.utc).isoformat()
training_contract = {
    "experiment": EXPERIMENT,
    "created_at": created_at,
    "role": "diagnose",
    "canonical_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "direct_comparison_baseline": "release16_joint_target_holdout_slice_repair_v1",
    "source_experiment": "release16_joint_target_holdout_slice_repair_v1",
    "selected_source": selected_source,
    "required_wf_windows": list(REQUIRED_FOLDS),
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "target_gate": bool(selected_score.get("target_gate", False)),
    "bc_holdout_slice_gate": bool(selected_score.get("holdout_slice_gate", False)),
    "holdout_repair_gate": bool(selected_score.get("holdout_repair_gate", False)),
    "target_contract_gate": bool(selected_score.get("holdout_repair_gate", False)),
    "training_allowed_next": False,
    "ppo_allowed_next": False,
    "checkpoint_screen_allowed_next": bool(summary_core.get("checkpoint_screen_allowed_next", False)),
    "actor_retention_allowed": False,
    "promotion_allowed": False,
    "required_repair_terms": summary_core.get("required_repair_terms", []),
    "next_mechanism": summary_core.get("next_mechanism"),
}
write_json(OUT_DIR / "training_contract.json", training_contract)

training_contract


{'experiment': 'release16_holdout_slice_repair_candidate_screen_v1',
 'created_at': '2026-06-15T10:28:41.318642+00:00',
 'role': 'diagnose',
 'canonical_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
 'direct_comparison_baseline': 'release16_joint_target_holdout_slice_repair_v1',
 'source_experiment': 'release16_joint_target_holdout_slice_repair_v1',
 'selected_source': 'release16_downtrend_drawdown_objective_screen_v1',
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 20.0,
 'target_gate': False,
 'bc_holdout_slice_gate': False,
 'holdout_repair_gate': False,
 'target_contract_gate': False,
 'training_allowed_next': False,
 'ppo_allowed_next': False,
 'checkpoint_screen_allowed_next': False,
 'actor_retention_allowed': False,
 'promotion_allowed': False,
 'required_repair_terms': ['checkpoint_objective_loss',
  'drawdown_state_aux_loss',
  'release_state_aux_loss',
  'trend_preservation_floor'],
 'next_m

In [4]:
failed_folds = sorted(
    fold for fold, row in selected_by_fold.items()
    if not (bool(row.get("holdout_slice_gate")) and (row.get("relative_gap_to_target_pp") == 0.0) and (row.get("max_dd_gap_to_target_pp") == 0.0))
)
worst_dd = max(float(row["selected_max_dd"]) for row in selected_by_fold.values() if row.get("selected_max_dd") is not None)
min_relative = min(float(row["selected_relative_return"]) for row in selected_by_fold.values() if row.get("selected_relative_return") is not None)
max_dd_gap = max(float(row.get("max_dd_gap_to_target_pp") or 0.0) for row in selected_by_fold.values())
max_relative_gap = max(float(row.get("relative_gap_to_target_pp") or 0.0) for row in selected_by_fold.values())

summary = {
    **training_contract,
    **summary_core,
    "decision": summary_core["decision"],
    "changed_layer": ["model_selection", "bc_warm_start", "selection"],
    "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "source_cards": SOURCE_CARDS,
    "source_decisions": {card: summaries[card].get("decision") for card in SOURCE_CARDS},
    "primary_blocker": "bc_candidate_or_slice_gate_failed",
    "next_action": summary_core.get("next_mechanism"),
    "failed_folds": failed_folds,
    "selected_by_fold": selected_by_fold,
    "selected_repair_loss": float(summary_core.get("selected_repair_loss", 0.0)),
    "worst_wf_max_drawdown_pct": worst_dd,
    "min_wf_relative_return_pp": min_relative,
    "max_drawdown_gap_to_target_pp": max_dd_gap,
    "max_relative_gap_to_target_pp": max_relative_gap,
    "sidecars": [
        "candidate_repair_rows.csv",
        "holdout_repair_scores.csv",
        "training_contract.json",
        "summary.json",
        "conclusion.md",
    ],
}
write_json(OUT_DIR / "summary.json", summary)

conclusion = f"""# Release16 Holdout Slice Repair Candidate Screen

Decision: `{summary['decision']}`.

Selected repair source: `{summary['selected_source']}` with repair loss `{summary['selected_repair_loss']:.2f}`. It is the closest current repair because it keeps all selected MaxDD values below `{TARGET_MAX_DD_PCT:.2f}%`, but it still fails the retained-model path: holdout repair gate `{summary['holdout_repair_gate']}`, holdout slice gate `{summary['bc_holdout_slice_gate']}`, and target gate `{summary['target_gate']}`.

The selected path still has min relative return `{summary['min_wf_relative_return_pp']:.2f}pp` versus target `>= {TARGET_RELATIVE_RETURN_PP:.2f}pp`; failed folds: `{', '.join(failed_folds)}`. No actor is retained, no PPO is allowed, and reward/checkpoint screening is not allowed until the state-conditioned action head or auxiliary state loss passes this holdout repair screen.

Required repair terms: {', '.join(f'`{term}`' for term in summary['required_repair_terms'])}.
"""
(OUT_DIR / "conclusion.md").write_text(conclusion, encoding="utf-8")

display(Markdown(conclusion))
summary


# Release16 Holdout Slice Repair Candidate Screen

Decision: `state_objective_holdout_repair_not_ready_no_training`.

Selected repair source: `release16_downtrend_drawdown_objective_screen_v1` with repair loss `110.60`. It is the closest current repair because it keeps all selected MaxDD values below `20.00%`, but it still fails the retained-model path: holdout repair gate `False`, holdout slice gate `False`, and target gate `False`.

The selected path still has min relative return `-5.60pp` versus target `>= 30.00pp`; failed folds: `wf_2020, wf_2022, wf_2023_2024`. No actor is retained, no PPO is allowed, and reward/checkpoint screening is not allowed until the state-conditioned action head or auxiliary state loss passes this holdout repair screen.

Required repair terms: `checkpoint_objective_loss`, `drawdown_state_aux_loss`, `release_state_aux_loss`, `trend_preservation_floor`.


{'experiment': 'release16_holdout_slice_repair_candidate_screen_v1',
 'created_at': '2026-06-15T10:28:41.318642+00:00',
 'role': 'diagnose',
 'canonical_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
 'direct_comparison_baseline': 'release16_joint_target_holdout_slice_repair_v1',
 'source_experiment': 'release16_joint_target_holdout_slice_repair_v1',
 'selected_source': 'release16_downtrend_drawdown_objective_screen_v1',
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 20.0,
 'target_gate': False,
 'bc_holdout_slice_gate': False,
 'holdout_repair_gate': False,
 'target_contract_gate': False,
 'training_allowed_next': False,
 'ppo_allowed_next': False,
 'checkpoint_screen_allowed_next': False,
 'actor_retention_allowed': False,
 'promotion_allowed': False,
 'required_repair_terms': ['checkpoint_objective_loss',
  'drawdown_state_aux_loss',
  'release_state_aux_loss',
  'trend_preservation_floor'],
 'next_m